In [27]:
#Importando Bibliotecas que estou utilizando para a análise
import pandas as pd
from ydata_profiling import ProfileReport
import os

In [28]:
#Caminho da Base de dados
caminho_dados = r"C:\Users\gabri\OneDrive\Belgeler\GitHub\Python\AnalisesRedebras\Referência\Dados\RelPedidoCli.xls"
df_comercial = pd.read_excel(caminho_dados)

# Sobrescreve o dataframe removendo a última linha (Linha total que o próprio ERP ja envia junto)
df_comercial = df_comercial.iloc[:-1]

In [29]:
## ALTERAÇÕES DE LIMPEZA E FORMATAÇÃO ##

#Modificando Número para Texto (Pedidos e N.F -> Precisam ser identificados como texto para a análise)
colunas_texto = ['PEDIDO', 'NF']
for col in colunas_texto:
    df_comercial[col] = (
        df_comercial[col].astype(str)
    )

#------------------------------------------------------------------------------#
# Forçamos as colunas a serem strings para limpeza manual 
# Formatando os Valores de Preços e Custos
colunas_preco = ['TOTAL CUSTO', 'TOTAL ITENS', 'TOTAL PEDIDO', 'QTDE VENDA']
for col in colunas_preco:
    df_comercial[col] = (
        df_comercial[col].astype(str)
    )

for col in colunas_preco:
    # 1. Troca a vírgula pelo ponto decimal (1200,50 -> 1200.50)
    df_comercial[col] = df_comercial[col].str.replace(',', '.', regex=False)
    # 2. Converte para número real (float)
    df_comercial[col] = pd.to_numeric(df_comercial[col], errors='coerce')
    # 3. Arredonda para 2 casas decimais
    df_comercial[colunas_preco] = df_comercial[colunas_preco].round(2)

#------------------------------------------------------------------------------#
#Formatando Colunas de Datas
# Modificando as colunas de Data para o tipo Data
colunas_data = ['DATA', 'NF. EMISSAO']

for col in colunas_data:
    df_comercial[col] = (
    pd.to_datetime(df_comercial[col], dayfirst=True, errors='coerce')
    )

#------------------------------------------------------------------------------#
#Limpeza de Colunas não necessárias para o projeto (Deletando Colunas)
colunas_desnecessarias = ['EMP', 'REPRESENTANTE', 'TIPO', 'INTEGRAÇÃO', 'OUTRAS DESP', 'VLR. FRETE', 'DT. RETIRADA', 'FRETE TYPE', 'FIN',
                           'TOTAL ITENS C/ IPI', 'TOTAL ITENS', 'MKP']

df_comercial = df_comercial.drop(columns=colunas_desnecessarias, errors='ignore')   

#------------------------------------------------------------------------------#
# Criação de colunas
# Lucro R$ (Para saber o lucro de cada venda em dinheiro)
df_comercial['LUCRO R$'] = (
    df_comercial['TOTAL PEDIDO'] - df_comercial['TOTAL CUSTO']
)
df_comercial['LUCRO R$'] = df_comercial['LUCRO R$'].round(2)

# Total de Vendas Mês (Para saber o total de vendas do mês, somando o valor de cada venda)
df_comercial['TOTAL VENDAS MÊS'] = df_comercial['TOTAL PEDIDO'].sum()

# Formatando a coluna de Data para o formato brasileiro (dd/mm/yyyy)
df_comercial['DATA'] = df_comercial['DATA'].dt.strftime('%d/%m/%Y')

# Formatando a coluna de Data NF. EMISSAO para o formato brasileiro (dd/mm/yyyy)
df_comercial['NF. EMISSAO'] = df_comercial['NF. EMISSAO'].dt.strftime('%d/%m/%Y')

##--------------------------------------##
# Calculando o MKP REAL de cada venda
# 1.Markup com o multiplicador real (Preço de Venda / Custo) para comparar com o MKP do ERP
df_comercial['MKP REAL'] = df_comercial['TOTAL PEDIDO'] / df_comercial['TOTAL CUSTO']

# 2.Calculando o MKP REAL em % (Multiplicador real - 1) * 100 para comparar com o MKP do ERP
df_comercial['MKP REAL %'] = (df_comercial['MKP REAL'] - 1) * 100

# 3.Tratamento de dados em caso de erro
df_comercial['MKP REAL %'] = df_comercial['MKP REAL %'].replace([float('inf'), float('-inf')], 0)  # Substitui inf e -inf por 0

# Preenchendo os valores nulos da coluna MKP com 0 (zero) e colocando o formato de porcentagem com 2 casas decimais
df_comercial['MKP REAL %'] = df_comercial['MKP REAL %'].fillna(0)
df_comercial['MKP REAL %'] = df_comercial['MKP REAL %'].round(2)

#Alterando nome da coluna Clientes
df_comercial = df_comercial.rename(columns={'NOME': 'CLIENTES'})



In [ ]:
# Criação de DataFrame Complementar para análise comercial por cidades do Brasil

df_analise_cidades = df_comercial.groupby('UF').agg({
    'PEDIDO': 'count',
    'TOTAL PEDIDO': 'sum',
    'LUCRO R$': 'sum',
    'MKP REAL %': 'mean'
}).reset_index()

# Renomeando as colunas do DataFrame de análise por cidades para melhor entendimento

df_analise_cidades = df_analise_cidades.rename(columns={
    'PEDIDO': 'QUANTIDADE DE PEDIDOS',
    'TOTAL PEDIDO': 'TOTAL FATURAMENTO',
    'LUCRO R$': 'LUCRO TOTAL',
    'MKP REAL %': 'MÉDIA DE MKP %'
})

#Calculos
# Calculando o Ticket Médio por cidade (Total Faturamento / Quantidade de Pedidos)

df_analise_cidades['TICKET MÉDIO'] = df_analise_cidades['TOTAL FATURAMENTO'] / df_analise_cidades['QUANTIDADE DE PEDIDOS']

In [40]:
df_analise_cidades

,UF,QUANTIDADE DE PEDIDOS,TOTAL FATURAMENTO,LUCRO TOTAL,MÉDIA DE MKP %
0,AM,3,4683.71,2220.40,105.316667
1,BA,4,17529.45,5730.34,83.080000
2,CE,5,8343.76,3228.52,186.136000
3,DF,3,68086.76,35209.67,88.810000
4,ES,8,15419.73,7972.56,122.068750
5,GO,2,3835.20,1203.45,44.520000
6,MA,1,2858.00,1071.76,60.000000
7,MG,23,107166.76,19929.48,81.130435
8,MT,1,11109.52,4757.88,74.910000
9,PA,2,2139.71,1203.11,136.920000


In [41]:
df_analise_cidades['TICKET MÉDIO'] = df_analise_cidades['TOTAL FATURAMENTO'] / df_analise_cidades['QUANTIDADE DE PEDIDOS']

In [42]:
df_analise_cidades

,UF,QUANTIDADE DE PEDIDOS,TOTAL FATURAMENTO,LUCRO TOTAL,MÉDIA DE MKP %,TICKET MÉDIO
0,AM,3,4683.71,2220.40,105.316667,1561.236667
1,BA,4,17529.45,5730.34,83.080000,4382.362500
2,CE,5,8343.76,3228.52,186.136000,1668.752000
3,DF,3,68086.76,35209.67,88.810000,22695.586667
4,ES,8,15419.73,7972.56,122.068750,1927.466250
5,GO,2,3835.20,1203.45,44.520000,1917.600000
6,MA,1,2858.00,1071.76,60.000000,2858.000000
7,MG,23,107166.76,19929.48,81.130435,4659.424348
8,MT,1,11109.52,4757.88,74.910000,11109.520000
9,PA,2,2139.71,1203.11,136.920000,1069.855000
